# Pydantic AI agent

Goal: run the docs agent on Pydantic AI instead of toyaikit, with a second tool that fetches a
whole page once search has pointed at it.

## Load the API key

Walks up from this folder to the `.env` at the repo root, so it works at any depth.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Install pydantic-ai

Records it in `pyproject.toml`. After a first install, restart the kernel so the import further
down can find it.

In [ ]:
!uv add pydantic-ai

# Search tools

Goal: a search that returns short highlighted snippets, plus a way to read the full file behind
a snippet that looks useful.

## Build the index

Whole pages this time, no chunking - search only has to point at the right file, and `get_file`
fetches the full text.

In [ ]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

## Set up the highlighter

Returns a few short snippets around the matched words instead of whole pages, so search results
stay small. `evidently` joins the stop words because every page mentions it.

In [ ]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [ ]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {'evidently'}

highlighter = Highlighter(
    highlight_fields=['content'],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer='snowball', stop_words=stopwords)
)

## Filename lookup

Maps each filename to its full text, for `get_file`.

In [ ]:
file_index = {doc['filename']: doc['content'] for doc in parsed_docs}

## Wrap both tools in a class

Keeps the index, highlighter and file lookup on one object. The type hints and docstrings do real
work here: Pydantic AI builds each tool's schema from them, so there's no hand-written JSON
schema this time.

In [ ]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        """
        Initialize the SearchTools instance.

        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.

        Args:
            filename (str): The filename of the file to retrieve.

        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [ ]:
search_tools = SearchTools(index, highlighter, file_index)

## Agent instructions

Same three-round plan as before, but round two now adds `get_file` calls, so the answer is built
from full pages rather than snippets.

In [ ]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2–3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

# The agent

Goal: hand the tools and instructions to a Pydantic AI `Agent` and ask it a question.

## Collect the tools

`get_instance_methods` picks up every public method on the object - here `search` and `get_file`.
The commented-out line is the same list written by hand.

In [ ]:
from pydantic_ai import Agent
from toyaikit.tools import get_instance_methods

In [ ]:
# tools = [search_tools.search, search_tools.get_file]
tools = get_instance_methods(search_tools)

## Create the agent

The `openai:` prefix tells Pydantic AI which provider to use, and it reads `OPENAI_API_KEY` from
the environment loaded at the top.

In [ ]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

## Ask one question

`run` is async, so it gets awaited - Jupyter allows `await` directly in a cell.

In [ ]:
query = 'how do I use evidently to monitor my machine learning models?'

In [ ]:
result = await search_agent.run(query)

The banner is pydantic-ai printing its version and a logging tip on the first run - it isn't
part of the answer.

## Read the answer

In [ ]:
print(result.output)

## Keep the conversation

In [ ]:
messages = result.all_messages()

## Token usage

In [ ]:
result.usage()

Fails on the installed pydantic-ai (2.61): `usage` is now a property, not a method, so it's
`result.usage` without the brackets. The chat loop below makes the same call.

## Walk through the messages

Each message is a request or a response, and each holds parts - user prompt, tool call, tool
return or text. Printing them shows every round trip the agent made.

In [ ]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

# Follow-up questions

Goal: carry on the same conversation by passing the earlier messages back in.

## Ask a follow-up

`message_history` gives the agent the first conversation, which is why "show me the code" gets
read as code for monitoring models.

In [ ]:
new_prompt = 'show me the code'

In [ ]:
result2 = await search_agent.run(new_prompt, message_history=messages)

## Read the follow-up answer

In [ ]:
print(result2.output)

## Message printer

The loop from above as a function, so the cells below can reuse it.

In [ ]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

## Only the new messages

`new_messages()` leaves out the history that was passed in, so this shows just the follow-up's
own round trips.

In [ ]:
print_messages(result2.new_messages())

# Chat loop

Goal: a turn-by-turn chat that keeps the history and adds up token usage across turns.

## Running token total

`RunUsage` objects add together, so this starts empty and each turn's usage goes on top.

In [ ]:
from pydantic_ai import RunUsage

## Chat

Type `stop` to end. Each turn's new messages go back into `messages`, so the next turn sees the
whole chat.

In [ ]:
messages = []
usage = RunUsage()

while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())

## Total usage

In [ ]:
usage

# Watching tool calls live

Goal: print each tool call as it happens, instead of digging through the messages afterwards.

## Event callback

Pydantic AI streams events while the agent runs. This handler picks out the tool calls and prints
them with the agent's name - handy once more than one agent is involved.

In [ ]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

## Run with the callback

In [ ]:
callback = NamedCallback(search_agent) 

In [ ]:
result = await search_agent.run(
    "how do I build dashboads with evidently?",
    event_stream_handler=callback
)